# Question 5 — HPC / Colab constellation pipeline

Run this notebook on **Google Colab** (or any fat CPU node). Do not use a GPU runtime: every expensive step is `cv2.matchTemplate` on CPU.

**Colab runtime:** Runtime → Change runtime type → **CPU**, and pick **High-RAM** if you have it. A GPU will sit idle.

Previous Kaggle score: **0.306**. This version spends extra compute on the 55% of the metric that failed (name + figure recovery):

1. Finer NCC grid (5°, 7 scales) and 2× claim maps.
2. Propose each of the 48 drawings from independently detected bright stars, not only from NCC peaks (those peaks are often decoy copies).
3. Verify every drawing with a Hungarian assignment on the claim maps, ranked by chance-corrected high-claim excess so large drawings cannot win by offering more targets.
4. Spare-fill leftover nodes when the winner is clear (figure recovery scores points, not patch IDs).

Output: `submission_hpc.csv` in the project folder.

## 0. Clone this branch, then run

Full steps are in `README.md`. Short version:

1. Runtime → **CPU** (High-RAM if you have it). No GPU.
2. Run the next cell. It installs packages and clones branch `colab-hpc` into `/content` if the project is not already there.
3. Run the remaining cells in order. Last cell downloads `submission_hpc.csv`.

**Packages:** `opencv-python-headless`, `numpy`, `scipy`, `pandas`, `pillow`. No PyTorch / TensorFlow / CUDA.

`cache_v3/` is not on the branch. Leave `Q5_FINE = 1` so Colab rebuilds `cache_hpc/`.

In [ ]:
# Colab setup: install deps and clone this branch if needed.
import os, sys, subprocess
from pathlib import Path

%pip install -q opencv-python-headless numpy scipy pandas pillow

REPO = "https://github.com/rushilara/CVProject1-MerchantsOfSanRamon.git"
BRANCH = "colab-hpc"
IN_COLAB = "google.colab" in sys.modules

if Path("q5_hpc.py").exists():
    PROJECT = Path(".").resolve()
elif IN_COLAB:
    PROJECT = Path("/content/CVProject1-MerchantsOfSanRamon")
    if not (PROJECT / "q5_hpc.py").exists():
        subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, str(PROJECT)])
else:
    PROJECT = Path(".").resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

needed = [
    "q5_hpc.py", "q5_identify.py", "q5_constellation.py", "q5_search.py", "q5_synth.py",
    "participant/train_ground_truth.csv", "participant/sample_submission.csv",
    "participant/patterns", "participant/train", "participant/validation",
]
missing = [p for p in needed if not Path(p).exists()]
print("cwd", Path.cwd())
if missing:
    raise FileNotFoundError("Upload the project folder. Missing: " + ", ".join(missing))
print("project files ok")

In [ ]:
# MUST run before `import q5_hpc` — those flags are read at import time.
import os

os.environ["Q5_FINE"] = "1"          # 5° × 7-scale search + 2× claim maps
os.environ["Q5_WORKERS"] = str(min(8, os.cpu_count() or 2))  # Colab: 2–8; cluster: raise this
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENCV_NUM_THREADS"] = "1"

import numpy as np
import q5_hpc
from q5_identify import load_patterns

print("fine", q5_hpc.FINE, "workers", q5_hpc.WORKERS)
print("angles", len(q5_hpc.ANGLES), "scales", q5_hpc.SCALES, "cache", q5_hpc.CACHE)
patterns = load_patterns(q5_hpc.ROOT / "patterns")
print(len(patterns), "drawings")

## 1. Correlation sweep (hours on Colab, much less on a 32-core node)

Each 32×32 patch is correlated against its 3000×3000 sky at every angle/scale. We keep up to 24 peak locations and a map of the best correlation at every sky position.

With `Q5_FINE=1` this is about 3.5× the laptop search. Leave the cell running. Results are cached in `cache_hpc/`, so a reconnect can continue.

In [ ]:
q5_hpc.search_missing()
print("stage 1 ready")

## 2. Name the figure, sweep rules on labelled scenes

Proposals come from a geometric hash of bright stars plus near-tied patch peaks. Every drawing is verified; the score is high-claim excess over chance, then a size residual. The sweep picks the rule that scores both the 3 real training scenes and the 16 synthetic ones.

In [ ]:
scenes = q5_hpc.labelled_scenes()
print(len(scenes), "labelled scenes")
best = q5_hpc.sweep(patterns, scenes)
q5_hpc.PARAMS.update(best[0])
print("using", best[0])
rows = q5_hpc.score_prepared(best[6], dict(q5_hpc.PARAMS), verbose=True)
real = [m["total"] for k, m, _ in rows if k.startswith("train_")]
syn = [m["total"] for k, m, _ in rows if k.startswith("syn_")]
print(f"mean real {np.mean(real):.3f}  synthetic {np.mean(syn):.3f}")

## 3. Write `submission_hpc.csv`

Upload that file to Kaggle. On Colab the last line also tries to download it to your laptop.

In [ ]:
from pathlib import Path

out = Path("submission_hpc.csv")
q5_hpc.write_submission(patterns, out)
print("wrote", out.resolve())

if "google.colab" in __import__("sys").modules:
    from google.colab import files
    files.download(str(out))